
# Datenexploration und -vorverarbeitung

**Ziel:** Laden und Bereinigen der *NYC HVFHS (High Volume For-Hire Service)*-Daten im Parquet-Format mittels **Apache Spark** und Übertragung in eine **PostgreSQL-Datenbank**.
    


## Datenquelle und Laden

Die Quell-Parquet-Datei wird über:

```python
spark.read.parquet()
```

eingelesen; Spark erkennt Datentypen automatisch.

Jede Zeile repräsentiert eine Fahrt mit Angaben zu:

```text
hvfhs_license_num, PULocationID, DOLocationID, pickup_datetime, dropoff_datetime,
trip_miles, trip_time, base_passenger_fare, driver_pay
```

Durch die verteilte Verarbeitung können auch größere Datenmengen performant analysiert und gefiltert werden.
    


## Erste Exploration und Validierung

Mit:
```python
df.select("hvfhs_license_num").distinct()
df.select("PULocationID", "DOLocationID").distinct()
```
werden eindeutige Anbieter-Codes und Zonen-IDs ermittelt.

Diese Prüfung stellt sicher, dass alle relevanten Dimensionen vorhanden sind, bevor Daten in die Datenbank geschrieben werden.

Über Hilfsfunktionen wie:

```python
get_or_create_providers()
get_or_create_zones()
```

wird geprüft, ob Einträge bereits existieren — fehlende werden automatisch ergänzt.
    


## Datenbereinigung

**Spaltenumbennenung:**
```python
PULocationID → pu_location_id
DOLocationID → do_location_id
```

**Mapping:**  
Anbieter-Codes werden per **UDF** in numerische IDs überführt:

```python
hvfhs_license_num → provider_id
```

**Boolesche Flags:**  
Konvertierung von *Y/N* zu *True/False* für Spalten wie:

```python
shared_request_flag, wav_match_flag
```

**Filterung:**  
Ungültige oder unplausible Fahrten werden ausgeschlossen, z. B.:

```python
trip_miles <= 0
base_passenger_fare <= 0
driver_pay <= 0
```
Diese Schritte standardisieren das Schema und stellen sicher, dass nur qualitativ hochwertige Datensätze weiterverarbeitet werden.
    


## Transformation und Strukturierung

Eine geordnete Spaltenliste wird definiert, um die Zieltabelle **trips** klar zu strukturieren:

```python
trip_cols = [
    "provider_id", "pu_location_id", "do_location_id",
    "pickup_datetime", "dropoff_datetime",
    "trip_miles", "trip_time", "base_passenger_fare", "driver_pay"
]
```

DataFrame-Partitionierung verbessert die Performance beim Schreiben:

```python
df.repartition(args.partitions)
```

Durch Logging ist der gesamte Prozess nachvollziehbar.
    


## Laden in PostgreSQL

Die Verbindung erfolgt über **JDBC**, Zugangsdaten per CLI-Argument.

Bereinigte Datensätze werden batchweise in die Faktentabelle **trips** geschrieben.

Fehlende Informationen zu **Provider** oder **Zonen** werden vorab in den Dimensionstabellen:

```text
providers
taxi_zones
```

ergänzt, um referentielle Integrität zu gewährleisten.
    


## Ergebnis

Entstanden ist ein **bereinigter, konsistenter Datensatz** mit vollständigen Referenzen auf Anbieter- und Zonentabellen.

Alle Spalten folgen einer einheitlichen Namens- und Typkonvention:

```text
BOOLEAN, TIMESTAMP, DECIMAL
```

Das Ergebnis dient als Grundlage für **Analyse- und Dashboard-Komponenten (Streamlit)**, die auf diesen Daten aufbauen.
    